# End-to-end regression: building energy intensity

Second of three notebooks. The skeleton is the same as
[classification](01_classification_end_to_end.ipynb); what changes is what "a good model"
means. A regression target has a *scale*, so the metrics are in physical units, the errors
are not exchangeable across the range, and a single aggregate number can hide a model that
is excellent on easy rows and useless on the ones that cost money.

The target is the specific energy demand of a building stock in kWh per m² and year. The
generator builds in the properties that break naive workflows:

* a non-linear response (sine of an interaction, quadratic in age),
* **multiplicative** noise, so the residual spread grows with the level,
* a perfectly duplicated column, a pure-noise column, and mixed dtypes,
* missing values that are MAR on a *categorical* driver.

## What you will see

| Stage | SKSurrogate pieces |
| --- | --- |
| Profiling and collinearity diagnosis | `dataset_profile`, `api.eda` |
| Imputation chosen by measurement, with the encoding trap made explicit | `impute_partition`, scikit-learn imputers |
| Bounded pipeline search | `AML(preset="regression")`, `to_audit_dict()` |
| Evaluation in physical units, with bootstrap intervals | `mltrack.nested_cv_evaluation` |
| Residual and heteroscedasticity diagnostics | `mltrack.regression_diagnostics`, `plot_regression_diagnostics` |
| Prediction intervals with a coverage guarantee | `split_conformal_regression_interval`, `prediction_interval_diagnostics` |
| Attribution | `permutation_importance`, `plot_partial_dependence` |
| Monitoring, packaging, serving, report | `drift_report`, `prediction_distribution_report`, bundles, `api.reports` |

## How to run

```bash
jupyter lab examples/notebooks/02_regression_end_to_end.ipynb
```

Offline by default. `SKSURROGATE_NOTEBOOK_DATA=openml` switches to the OpenML
`wine_quality` frame instead, which keeps the workflow and changes the target and units.
Artifacts land in `regression-report/`.

Everything here is a methods demonstration on synthetic data with a small search budget.

In [ ]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
NOTEBOOK_DIR = REPO_ROOT / "examples" / "notebooks"
if not (NOTEBOOK_DIR / "notebook_data.py").exists():
    NOTEBOOK_DIR = REPO_ROOT  # launched from inside examples/notebooks
sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (opt-in import)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.metrics import get_scorer, mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import SKSurrogate
from SKSurrogate import (
    DeploymentApprovalGate,
    FoldLocalDataPreprocessor,
    ModelBundle,
    ModelRegistry,
    assert_bundle_quality,
    dataset_profile,
    drift_report,
    load_bundle,
    mltrack,
    predict_batch,
    prediction_distribution_report,
    prediction_interval_diagnostics,
    save_bundle,
    split_conformal_regression_interval,
)
from api.eda import effect_sizes, formal_tests, missing_mechanism, stability
from api.preprocessing import impute_partition, missing_overview
from notebook_data import SEED, SOURCE, artifact_dir, regression_data
from notebook_report import compile_pdf, figure_of, figure_pdf, markdown_table, save_figure

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X has feature names")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

REPORT_DIR = artifact_dir("regression-report")
print(f"SKSurrogate {SKSurrogate.__version__} from {Path(SKSurrogate.__file__).parent}")
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"seed={SEED}  data source={SOURCE!r}  artifacts -> {REPORT_DIR.resolve()}")

## 1. The data

Same protocol as the classification notebook: print the provenance before looking at any
number, so every later table can be traced to a generator, a seed and a row count.

In [ ]:
dataset = regression_data()
frame = dataset.frame
TARGET = dataset.target

provenance = pd.DataFrame(
    [
        ("source", dataset.source),
        ("description", dataset.description),
        ("rows x columns", f"{frame.shape[0]} x {frame.shape[1]}"),
        ("target", TARGET),
        ("target mean / sd", f"{frame[TARGET].mean():.1f} / {frame[TARGET].std(ddof=1):.1f}"),
        ("target range", f"{frame[TARGET].min():.1f} .. {frame[TARGET].max():.1f}"),
        ("categorical features", ", ".join(frame.select_dtypes(exclude=[np.number]).columns) or "none"),
        ("missing cells %", f"{frame.isna().to_numpy().mean() * 100:.2f}"),
    ],
    columns=["field", "value"],
)
display(provenance)
for key, value in dataset.notes.items():
    print(f"{key}: {value}")
display(frame.head())

## 2. Profile

Two things in this profile decide the whole rest of the notebook:

* `duplicate_features` — `gross_area_m2` is `floor_area_m2` exactly, so the numeric block
  is rank deficient and any linear model is singular;
* `multicollinearity.condition_number` is `None` with `rank_deficient`, which is the
  numeric signature of the same fact.

The association table is also worth reading carefully for a regression target: Pearson
correlation only sees *linear* association, so a strong sine or quadratic effect can look
weak. That is a reason to look at binned profiles, not a reason to drop the feature.

In [ ]:
profile = dataset_profile(
    frame,
    target=TARGET,
    group_by="heating_system",
    max_rows=2000,
    max_features=50,
    max_categories=10,
    random_state=SEED,
)

columns_table = pd.DataFrame(profile["columns"])[
    ["column", "dtype", "kind", "unique_count", "missing_count", "missing_rate"]
]
display(columns_table)

association_table = pd.DataFrame(profile["target_associations"])
display(association_table.head(10))

quality = profile["data_quality"]
print("duplicate features :", quality["duplicate_features"])
print("constant features  :", quality["constant_features"])
mcl = profile["multicollinearity"]
print("condition number   :",
      mcl["condition_number"] if mcl["condition_number"] is not None
      else f"rank deficient ({mcl['condition_number_reason'].replace('_', ' ')})")
print("largest VIF        :", max((row["value"] or 0.0) for row in mcl["vif"]))

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4.4), constrained_layout=True)

axes[0].hist(frame[TARGET], bins=40, color="C0", alpha=0.85)
axes[0].axvline(frame[TARGET].mean(), color="C3", label=f"mean {frame[TARGET].mean():.1f}")
axes[0].axvline(frame[TARGET].median(), color="k", linestyle="--",
                label=f"median {frame[TARGET].median():.1f}")
axes[0].set(xlabel=f"{TARGET} (kWh/m2/yr)", ylabel="buildings", title="Target distribution")
axes[0].legend(fontsize=8)

missing_pct = (100 * frame.isna().mean()).sort_values()
missing_pct = missing_pct[missing_pct > 0]
axes[1].barh(missing_pct.index, missing_pct.to_numpy(), color="C1")
axes[1].set(xlabel="missing cells (% of rows)", title="Where the gaps are")
axes[1].tick_params(axis="y", labelsize=9)

top_assoc = (
    association_table.assign(strength=lambda d: d["value"].abs())
    .nlargest(10, "strength")
    .sort_values("strength")
)
axes[2].barh(top_assoc["feature"], top_assoc["strength"].to_numpy(), color="C2")
axes[2].set(xlabel="|Pearson r| with target", title="Linear association only")
axes[2].tick_params(axis="y", labelsize=9)

figure.suptitle("Regression frame: target shape, gaps, linear associations", fontsize=13)
save_figure(figure, REPORT_DIR, "eda-overview")
display(figure)

The Pearson panel is the one to distrust. `insulation_score` enters the response through
`sin(pi * insulation * glazing)` and `building_age_years` through a quadratic, so their
marginal correlations understate both. The search and the partial-dependence section will
show the effects the correlation misses.

## 3. Missing data

`setpoint_c` is the interesting column: its missingness depends on the *heating system*,
a categorical variable. `missing_mechanism` reports the numeric shift table, which cannot
see a categorical driver directly — so the figure below shows the mechanism the table
cannot: the missing rate of `setpoint_c` by heating system.

In [ ]:
display(pd.DataFrame([missing_overview(frame, target=TARGET, ignored=[])]))
patterns = pd.DataFrame(profile["missingness_patterns"]["patterns"])
patterns["missing_features"] = patterns["missing_features"].apply(
    lambda items: ", ".join(items) if items else "(complete)"
)
display(patterns)

mechanism = missing_mechanism(frame, target=TARGET)
print("Little MCAR test:", json.dumps(mechanism["mcar"]))
display(pd.DataFrame(mechanism["shifts"]).nlargest(8, "standardized_difference"))

In [ ]:
setpoint_missing = (
    frame.assign(setpoint_missing=frame["setpoint_c"].isna())
    .groupby(frame["heating_system"].fillna("__missing__"))["setpoint_missing"]
    .agg(["mean", "size"])
    .reset_index()
)
display(setpoint_missing.round(3))

figure, ax = plt.subplots(figsize=(7.2, 3.6), constrained_layout=True)
ax.bar(setpoint_missing["heating_system"], setpoint_missing["mean"], color="C3")
for row in setpoint_missing.itertuples():
    ax.text(row.heating_system, row.mean + 0.004, f"n={row.size}", ha="center", fontsize=9)
ax.set(xlabel="heating system", ylabel="share of rows with setpoint_c missing",
       title="MAR on a categorical driver: setpoint readings by system type")
save_figure(figure, REPORT_DIR, "missing-mechanism")
display(figure)

## 4. Formal tests, effect sizes, stability

The same three diagnostics as in the classification notebook, with one change in reading:
`effect_sizes` groups by `heating_system`, and the standardised differences are now about
*energy* between system types. They still are not causal — system type correlates with
building age and fabric, and the generator made it so.

In [ ]:
tests_table = pd.DataFrame(formal_tests(frame, target=TARGET, alpha=0.05)["columns"])
display(tests_table.sort_values("statistic", ascending=False))

effects = effect_sizes(
    frame.dropna(subset=[TARGET]),
    target=TARGET,
    group_by="heating_system",
    bootstrap=200,
    random_state=SEED,
)
effects_table = pd.DataFrame(effects["comparisons"])
display(effects_table.assign(abs_d=lambda d: d["cohens_d"].abs()).nlargest(6, "abs_d").drop(columns="abs_d"))
print(effects["interpretation"])

stability_report = stability(frame, target=TARGET, random_state=SEED)
display(
    pd.DataFrame(stability_report["columns"])
    .sort_values("max_relative_difference", ascending=False)
    .head(6)
)
print("largest half-to-half relative difference:",
      round(stability_report["max_relative_difference"], 4))

## 5. Prune the duplicate and the identifier, then re-run the test that could not run

Two columns leave the frame here, for two different reasons.

* `gross_area_m2` is an exact copy of `floor_area_m2`. Little's MCAR test needs a
  non-singular covariance estimate, and with the duplicate present the EM step fails
  (`em_did_not_converge`) — the diagnostic reporting a data problem before you found it.
* `meter_serial` is an identifier: unique on every row, no signal, and a tree can memorise
  rows with it. Name-based detection only catches ids whose name says so
  (`quality_flags(...)["identifier_columns"]` is empty here), so the *shape* of the column
  is checked instead — an integer or string column unique on practically every row.

In [ ]:
def describe_condition(number):
    # None means the design matrix was rank deficient (perfect collinearity).
    return "rank deficient" if number is None else f"{number:.3f}"


# An identifier is a lookup key, not a feature: it carries no signal and lets a tree
# memorise rows. Name-based detection only catches obvious ids, so detect the *shape*
# instead — an integer or string column that is unique on practically every row.
def looks_like_identifier(series, row_count):
    # A key is an integer (or string) column whose values are distinct row by row: at
    # least half of the rows carry a new level for integers, practically every row for
    # strings. Continuous float columns are measurements and are never dropped here --
    # a float that is unique on every row is still a measurement.
    if series.dtype.kind in "iu":
        threshold = 0.5
    elif series.dtype == object:
        threshold = 0.98
    else:
        return False
    distinct = series.nunique(dropna=True)
    return distinct > 20 and distinct / row_count > threshold


identifier_like = [
    column
    for column in frame.columns
    if column != TARGET and looks_like_identifier(frame[column], len(frame))
]
print("identifier-like columns:", identifier_like)
frame = frame.drop(columns=["gross_area_m2", *identifier_like])
reprofiled = dataset_profile(frame, target=TARGET, max_rows=2000, random_state=SEED)
print("condition number:", describe_condition(mcl["condition_number"]),
      "->", describe_condition(reprofiled["multicollinearity"]["condition_number"]))
print("duplicates left :", reprofiled["data_quality"]["duplicate_features"])
mechanism_pruned = missing_mechanism(frame, target=TARGET)
print("Little MCAR test after pruning:", json.dumps(mechanism_pruned["mcar"]))

## 6. Imputation, measured — and a trap worth meeting

Same protocol as before: hide 150 observed `setpoint_c` values, impute, score the
reconstruction. Two variants are added because this frame has a categorical driver:

* **capacity columns** — one-hot indicators for the heating system are part of the
  imputation matrix. Without them the strongest predictor of the setpoint is simply absent
  (only numeric columns would be used), which is exactly what a careless pipeline does.
* **kNN, raw vs standardised** — the scale trap from the classification notebook, repeated
  here because it bites harder when one column spans 40 and another spans 4e-1.

The result decides the method for the materialised frame: `iterative`, on the encoded matrix.

In [ ]:
# The imputation matrix: numeric columns plus one-hot indicators for the categorical
# driver. This is what FoldLocalDataPreprocessor would build inside the pipeline.
imputation_matrix = pd.concat(
    [
        frame.select_dtypes(include=[np.number]).drop(columns=[TARGET]),
        pd.get_dummies(frame["heating_system"].fillna("__missing__"), dtype=float),
    ],
    axis=1,
)

rng = np.random.default_rng(SEED)
observed_rows = np.flatnonzero(imputation_matrix["setpoint_c"].notna().to_numpy())
held_out = np.sort(rng.choice(observed_rows, size=150, replace=False))
truth = imputation_matrix.loc[held_out, "setpoint_c"].to_numpy()

masked = imputation_matrix.copy()
masked.loc[held_out, "setpoint_c"] = np.nan
column_index = list(imputation_matrix.columns).index("setpoint_c")


def reconstruction_error(imputed_column):
    residual = np.asarray(imputed_column) - truth
    return float(np.sqrt(np.mean(residual ** 2))), float(np.mean(np.abs(residual)))


rows = []
for method, imputer in (
    ("median", SimpleImputer(strategy="median")),
    ("iterative (MICE-style)", IterativeImputer(max_iter=10, random_state=SEED)),
    ("knn k=5 (raw scale)", KNNImputer(n_neighbors=5)),
):
    filled = imputer.fit_transform(masked)
    rmse, mae = reconstruction_error(filled[held_out, column_index])
    rows.append({"method": method, "rmse": rmse, "mae": mae, "rows_kept": len(masked)})

scaler = StandardScaler()
scaled = pd.DataFrame(scaler.fit_transform(masked), columns=masked.columns)
filled_scaled = scaler.inverse_transform(KNNImputer(n_neighbors=5).fit_transform(scaled))
rmse, mae = reconstruction_error(filled_scaled[held_out, column_index])
rows.append({"method": "knn k=5 (standardised)", "rmse": rmse, "mae": mae, "rows_kept": len(masked)})

rows.append({"method": "drop_rows", "rmse": np.nan, "mae": np.nan,
             "rows_kept": int(masked.dropna().shape[0])})
imputation_table = pd.DataFrame(rows).sort_values("rmse", na_position="last")
display(imputation_table.round(3))

figure, ax = plt.subplots(figsize=(8.4, 3.8), constrained_layout=True)
plotted = imputation_table.dropna(subset=["rmse"])
ax.barh(plotted["method"], plotted["rmse"], color="C0", label="RMSE")
ax.barh(plotted["method"], plotted["mae"], height=0.45, color="C3", label="MAE")
ax.invert_yaxis()
ax.set(xlabel="reconstruction error on 150 hidden setpoint_c values (kWh-relevant units: degC)")
ax.set_title("Imputation quality on a column with a categorical driver")
ax.legend()
save_figure(figure, REPORT_DIR, "imputation-benchmark")
display(figure)
print(f"drop_rows would keep {imputation_table.iloc[-1]['rows_kept']} of {len(masked)} rows")

Read the size of the effect: the conditional methods are an order of magnitude better
than the marginal median, because the setpoint is mostly the heating system plus a little
age and fabric. An imputer that does not see the category cannot know that, and no amount
of tuning fixes it — the fix is *encoding before imputing*, which is what
`FoldLocalDataPreprocessor` does by construction.

## 7. Leakage: whole-frame imputation versus fold-local preprocessing

Because the imputer here learns a *conditional model* (MICE fits a regressor per column),
fitting it on the full frame and then cross-validating is genuine leakage: each fold's
holdout rows contributed to the imputation model that produced their own features.

The two scores below will be close, and the fold-local one is the one that is valid.

In [ ]:
leaky_frame = pd.concat(
    [
        pd.DataFrame(
            IterativeImputer(max_iter=10, random_state=SEED).fit_transform(
                frame.select_dtypes(include=[np.number]).drop(columns=[TARGET])
            ),
            columns=frame.select_dtypes(include=[np.number]).drop(columns=[TARGET]).columns,
        ),
        frame[[TARGET]].reset_index(drop=True),
    ],
    axis=1,
)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

leaky_score = cross_val_score(
    RandomForestRegressor(n_estimators=150, random_state=SEED),
    leaky_frame.drop(columns=[TARGET]),
    leaky_frame[TARGET],
    cv=cv,
    scoring="r2",
    n_jobs=1,
)
safe_score = cross_val_score(
    make_pipeline(FoldLocalDataPreprocessor(), RandomForestRegressor(n_estimators=150, random_state=SEED)),
    frame.drop(columns=[TARGET]),
    frame[TARGET],
    cv=cv,
    scoring="r2",
    n_jobs=1,
)
comparison = pd.DataFrame(
    [
        {"protocol": "impute whole frame, then 5-fold CV", "cv_r2_mean": leaky_score.mean(),
         "cv_r2_std": leaky_score.std(ddof=1), "leakage_safe": False},
        {"protocol": "FoldLocalDataPreprocessor inside each fold", "cv_r2_mean": safe_score.mean(),
         "cv_r2_std": safe_score.std(ddof=1), "leakage_safe": True},
    ]
)
display(comparison.round(4))
print(f"difference in R2: {abs(leaky_score.mean() - safe_score.mean()):.4f} "
      "(small, and irrelevant: the leaky protocol is invalid whatever it scores)")

## 8. The modelling frame

Split first, then fit the imputer on the training half only, then encode the categorical
with a train-derived vocabulary. The materialised frames are numeric and complete because
the tracker's explanation paths consume numeric matrices.

In [ ]:
train_raw, validation_raw = train_test_split(
    frame, test_size=0.30, random_state=SEED
)
train_raw = train_raw.reset_index(drop=True)
validation_raw = validation_raw.reset_index(drop=True)

numeric_columns = [
    column for column in frame.columns
    if column != TARGET and frame[column].dtype != object
]
categorical_columns = [
    column for column in frame.columns
    if column != TARGET and frame[column].dtype == object
]

train_dummies = pd.get_dummies(train_raw[categorical_columns].fillna("__missing__"), dtype=int)
imputer = IterativeImputer(max_iter=10, random_state=SEED).fit(
    pd.concat([train_raw[numeric_columns], train_dummies], axis=1)
)


def build_partition(partition):
    # One-hot with the TRAIN vocabulary, impute with the TRAIN-fitted imputer (the
    # indicators are part of the imputation matrix — section 6 showed that matters),
    # then keep the numeric block plus the indicators.
    dummies = pd.get_dummies(partition[categorical_columns].fillna("__missing__"), dtype=int)
    dummies = dummies.reindex(columns=train_dummies.columns, fill_value=0).reset_index(drop=True)
    combined = pd.concat([partition[numeric_columns].reset_index(drop=True), dummies], axis=1)
    imputed = pd.DataFrame(imputer.transform(combined), columns=combined.columns)
    numeric_out = imputed[numeric_columns]
    return pd.concat(
        [numeric_out, dummies, partition[[TARGET]].reset_index(drop=True)], axis=1
    )


train = build_partition(train_raw)
validation = build_partition(validation_raw)

feature_columns = [column for column in train.columns if column != TARGET]
assert not train[feature_columns].isna().any().any()
assert not validation[feature_columns].isna().any().any()
assert feature_columns == [c for c in validation.columns if c != TARGET]

print(f"train {train.shape} | validation {validation.shape}")
print("features:", ", ".join(feature_columns))
print(f"target: train {train[TARGET].mean():.1f} +/- {train[TARGET].std(ddof=1):.1f} | "
      f"validation {validation[TARGET].mean():.1f} +/- {validation[TARGET].std(ddof=1):.1f}")

And the same decision through the one-shot API that the Data Preprocessing page uses —
restricted to the training partition, as it must be:

In [ ]:
imputed_copy, imputation_report = impute_partition(
    train_raw,
    target=TARGET,
    ignored=[],
    method="iterative",
    columns=[column for column in numeric_columns if train_raw[column].isna().any()],
    params={"max_iter": 10},
)
print(json.dumps({key: imputation_report[key] for key in
                  ("method", "columns", "rows_dropped", "cells_imputed")}, indent=2, default=str))

## 9. Register and search

`scoring="neg_root_mean_squared_error"` — the preset default is already a regression
metric, but stating it explicitly keeps the choice visible. `KFold` on the training
partition, shuffed with a fixed seed.

In [ ]:
workdir = artifact_dir(".notebook-work", root=REPORT_DIR)
tracker = mltrack("notebook-regression", db_name=str(workdir / "mltrace.db"))
tracker.RegisterData(train, TARGET, partition="train")
tracker.RegisterData(validation, TARGET, partition="validation")
tracker.SetCV(KFold(n_splits=5, shuffle=True, random_state=SEED))
print("task:", tracker.task, "| cv:", tracker.GetCVSpec())

In [ ]:
from SKSurrogate import AML

search = AML(
    preset="regression",
    scoring="neg_root_mean_squared_error",
    max_evals=12,
    time_limit=90,
    cv=4,
    random_state=SEED,
    n_jobs=1,
    check_point=str(workdir) + "/",
    verbose=0,
)
search.fit(train[feature_columns].to_numpy(), train[TARGET].to_numpy())

audit = search.to_audit_dict()
# The audit stores the score oriented so that larger is better for the metric family;
# for RMSE that means the reported number is the error itself, in kWh/m2/yr.
best_cv_rmse = audit["best_candidate"]["score"]
print("termination:", audit["termination"])
print("budget:", {k: audit["budget"][k] for k in ("max_evals", "time_limit", "evaluations", "elapsed_seconds")})
print(f"best CV RMSE: {best_cv_rmse:.3f} kWh/m2/yr")
print("best pipeline:", audit["best_candidate"]["pipeline"])

In [ ]:
trials = []
for candidate in audit["candidates"]:
    family = candidate["pipeline"][0].rsplit(".", 1)[-1] if candidate["pipeline"] else "unknown"
    for stage in candidate["stages"]:
        for record in stage["evaluations"]:
            trials.append(
                {
                    "family": family,
                    "trial": record["trial"],
                    "status": record["status"],
                    "params": record["params"],
                    "cv_rmse": record["score"],
                    "duration_s": record["duration"],
                }
            )
trial_table = pd.DataFrame(trials)
trial_table["cumulative_s"] = trial_table.groupby("family")["duration_s"].cumsum()
display(trial_table.nsmallest(6, "cv_rmse")[["family", "trial", "cv_rmse", "duration_s"]])

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
for family, group in trial_table.groupby("family"):
    order = group.sort_values("trial")
    axes[0].plot(order["trial"], order["cv_rmse"], marker="o", label=family)
    axes[1].step(order["cumulative_s"], order["cv_rmse"].cummin(), where="post", label=family)
axes[0].set(xlabel="trial", ylabel="4-fold CV RMSE (lower is better)", title="Search trace")
axes[1].set(xlabel="cumulative fit seconds", ylabel="best CV RMSE so far", title="Budget spent")
for ax in axes:
    ax.legend(fontsize=8)
figure.suptitle("AML search on a continuous target", fontsize=13)
save_figure(figure, REPORT_DIR, "search-trace")
display(figure)

## 10. Evaluation in physical units

Three numbers, three different questions:

* **nested CV R²** — how much of the variance the *procedure* explains, with selection
  inside each outer fold;
* **MAE / RMSE in kWh/m²/yr** — what an operator actually budgets for;
* **the dummy baseline** — a constant prediction at the training mean, which is the floor
  any "model" must clear.

Bootstrap intervals resample validation rows. They quantify sampling variability of this
estimate on this partition; they say nothing about next year's buildings.

In [ ]:
evaluation = tracker.nested_cv_evaluation(
    search.best_estimator_,
    inner_cv=3,
    outer_cv=4,
    train_partition="train",
    validation_partition="validation",
    scoring=get_scorer("r2"),
    repeats=2,
    confidence_level=0.95,
)
best_model = search.best_estimator_
validation_predictions = best_model.predict(validation[feature_columns])
y_validation = validation[TARGET].to_numpy()

dummy = DummyRegressor(strategy="mean").fit(train[feature_columns], train[TARGET])
dummy_predictions = dummy.predict(validation[feature_columns])


def bootstrap_metrics(y_true, y_pred, draws=1000, seed=SEED):
    # Percentile bootstrap over validation rows for R2, MAE and RMSE.
    rng = np.random.default_rng(seed)
    collected = {"r2": [], "mae": [], "rmse": []}
    n = len(y_true)
    for _ in range(draws):
        index = rng.integers(0, n, n)
        collected["r2"].append(r2_score(y_true[index], y_pred[index]))
        collected["mae"].append(mean_absolute_error(y_true[index], y_pred[index]))
        collected["rmse"].append(root_mean_squared_error(y_true[index], y_pred[index]))
    return {key: np.percentile(values, [2.5, 97.5]) for key, values in collected.items()}


metric_intervals = bootstrap_metrics(y_validation, validation_predictions)
metric_table = pd.DataFrame(
    [
        {
            "model": "AML best (validation)",
            "r2": r2_score(y_validation, validation_predictions),
            "mae": mean_absolute_error(y_validation, validation_predictions),
            "rmse": root_mean_squared_error(y_validation, validation_predictions),
        },
        {
            "model": "dummy mean (validation)",
            "r2": r2_score(y_validation, dummy_predictions),
            "mae": mean_absolute_error(y_validation, dummy_predictions),
            "rmse": root_mean_squared_error(y_validation, dummy_predictions),
        },
    ]
)
interval_table = pd.DataFrame(
    [
        {"metric": metric, "estimate": value, "lower": metric_intervals[metric][0],
         "upper": metric_intervals[metric][1], "bootstrap_draws": 1000}
        for metric, value in (
            ("r2", metric_table.loc[0, "r2"]),
            ("mae", metric_table.loc[0, "mae"]),
            ("rmse", metric_table.loc[0, "rmse"]),
        )
    ]
)
display(metric_table.round(3))
display(interval_table.round(3))
cv_interval = evaluation["confidence_interval"]
print(f"nested-CV mean R2 {evaluation['mean_outer_score']:.3f} "
      f"(CI {cv_interval['lower']:.3f}-{cv_interval['upper']:.3f})")
print(f"held-out R2       {metric_table.loc[0, 'r2']:.3f} "
      f"(bootstrap {metric_intervals['r2'][0]:.3f}-{metric_intervals['r2'][1]:.3f})")
overlaps = not (
    metric_intervals["r2"][1] < cv_interval["lower"]
    or metric_intervals["r2"][0] > cv_interval["upper"]
)
print("the two intervals overlap:", overlaps)
print("held-out beats the dummy mean baseline:", metric_table.loc[0, "r2"] > metric_table.loc[1, "r2"])

In [ ]:
repeats = np.asarray(evaluation["repeated_outer_scores"], dtype=float)
interval = evaluation["confidence_interval"]

figure, ax = plt.subplots(figsize=(9.2, 3.6), constrained_layout=True)
ax.axvspan(interval["lower"], interval["upper"], alpha=0.18, color="C0",
           label=f"{interval['confidence_level']:.0%} CI of repeat means (nested CV)")
ax.scatter(repeats, np.arange(1, len(repeats) + 1), color="C0", zorder=3, label="repeat-level mean")
ax.errorbar(
    metric_table.loc[0, "r2"],
    len(repeats) + 1.35,
    xerr=[[metric_table.loc[0, "r2"] - metric_intervals["r2"][0]],
          [metric_intervals["r2"][1] - metric_table.loc[0, "r2"]]],
    fmt="s", color="C3", capsize=5, label="held-out R2 (95% bootstrap CI)",
)
ax.set(yticks=[*range(1, len(repeats) + 1), len(repeats) + 1.35], ylabel="estimate", xlabel="R2")
ax.set_yticklabels([f"repeat {i}" for i in range(1, len(repeats) + 1)] + ["held-out"])
ax.set_title("Generalisation estimate on two scopes")
ax.legend(fontsize=8, loc="lower right")
save_figure(figure, REPORT_DIR, "evaluation")
display(figure)

## 11. Residuals: the part a regression report usually skips

The tracker's diagnostic panel is the UI view (its internal unseeded split, as in the
classification notebook). The panel after it is the authoritative version on the held-out
partition, and it answers the question the aggregate metrics hide: **where** is the model
wrong?

The generator made the noise multiplicative, so a constant-width error bar is the wrong
mental model. Plotting residual spread in bins of the prediction makes that visible
immediately, and it is the reason the next section uses a conformal interval instead of a
±2·σ band.

In [ ]:
logged_id = tracker.LogModel(best_model, "aml-regressor")
plt.close("all")
figure = figure_of(tracker.plot_regression_diagnostics(logged_id, bins=24))
save_figure(figure, REPORT_DIR, "regression-diagnostics")
display(figure)

residuals = y_validation - validation_predictions
print("tracker regression_diagnostics:", {
    key: round(value, 3) for key, value in tracker.regression_diagnostics(logged_id).items()
    if isinstance(value, (int, float))
})

In [ ]:
from scipy import stats

# Binned residual spread: the heteroscedasticity check.
deciles = pd.qcut(validation_predictions, 8, labels=False, duplicates="drop")
spread = (
    pd.DataFrame({"bin": deciles, "prediction": validation_predictions, "residual": residuals})
    .groupby("bin")
    .agg(prediction_mean=("prediction", "mean"), residual_sd=("residual", "std"),
         residual_mae=("residual", lambda values: np.mean(np.abs(values))), n=("residual", "size"))
    .reset_index()
)
display(spread.round(2))

# Error by predicted decile, expressed as a percentage of the prediction.
decile_columns = pd.qcut(validation_predictions, 10, labels=False, duplicates="drop")
error_by_decile = (
    pd.DataFrame({"decile": decile_columns + 1, "prediction": validation_predictions,
                  "residual": residuals})
    .groupby("decile")
    .agg(prediction_mean=("prediction", "mean"),
         mae=("residual", lambda values: np.mean(np.abs(values))),
         n=("residual", "size"))
)
error_by_decile["mae_pct_of_prediction"] = 100 * error_by_decile["mae"] / error_by_decile["prediction_mean"]
display(error_by_decile.round(2))

figure, axes = plt.subplots(2, 2, figsize=(12.6, 8.2), constrained_layout=True)
axes[0, 0].scatter(y_validation, validation_predictions, s=16, alpha=0.7)
limits = [min(y_validation.min(), validation_predictions.min()),
          max(y_validation.max(), validation_predictions.max())]
axes[0, 0].plot(limits, limits, "k--", linewidth=1, label="identity")
axes[0, 0].set(xlabel="actual (kWh/m2/yr)", ylabel="predicted", title="Predicted vs actual (validation)")
axes[0, 0].legend(fontsize=8)

axes[0, 1].scatter(validation_predictions, residuals, s=16, alpha=0.7)
axes[0, 1].axhline(0, color="k", linewidth=1)
axes[0, 1].plot(spread["prediction_mean"], spread["residual_sd"], "o-", color="C3",
                label="binned residual SD")
axes[0, 1].plot(spread["prediction_mean"], -spread["residual_sd"], "o-", color="C3", alpha=0.6)
axes[0, 1].set(xlabel="predicted", ylabel="residual", title="Residuals vs predicted (heteroscedastic)")
axes[0, 1].legend(fontsize=8)

stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Residual normal Q-Q (descriptive)")

axes[1, 1].bar(error_by_decile.index.astype(int), error_by_decile["mae"], color="C2")
axes[1, 1].set(xlabel="predicted-value decile (1 = lowest demand)",
               ylabel="MAE (kWh/m2/yr)", title="Absolute error grows with the level")
figure.suptitle("Where the model is wrong, on the held-out partition", fontsize=13)
save_figure(figure, REPORT_DIR, "residual-diagnostics")
display(figure)

## 12. Prediction intervals with a coverage statement

A point forecast of energy demand is rarely what a decision needs — the question is usually
"how bad could it be". Split conformal prediction turns any regressor into an interval with
a finite-sample *marginal* coverage guarantee, provided

1. the calibration rows are exchangeable with the future rows, and
2. the model was fitted without them.

So the model is refitted on a fit slice and calibrated on a disjoint slice, both taken from
the training partition. The validation partition is only used to *check* coverage.

Then the honest limitation: marginal coverage is not conditional coverage. With
multiplicative noise a constant-width interval is too wide for efficient buildings and too
narrow for inefficient ones. The **normalised** variant scales the residual quantile by
the prediction, and the decile table shows what that buys.

In [ ]:
fit_X, calibration_X, fit_y, calibration_y = train_test_split(
    train[feature_columns], train[TARGET], test_size=0.30, random_state=SEED
)

conformal = split_conformal_regression_interval(
    clone(best_model),
    fit_X,
    fit_y,
    calibration_X,
    calibration_y,
    validation[feature_columns],
    alpha=0.10,
)
print("method:", conformal["method"])
print("quantile (kWh/m2/yr):", round(conformal["quantile"], 3),
      "| calibration rows:", conformal["calibration_size"])

coverage_report = prediction_interval_diagnostics(
    y_validation, conformal["lower"], conformal["upper"], alpha=0.10
)
print(json.dumps({key: coverage_report[key] for key in
                  ("sample_size", "empirical_coverage", "nominal_coverage", "mean_interval_width",
                   "mean_interval_score", "coverage_interval")}, indent=2, default=str))

# Normalised (locally adaptive) split conformal: quantile of |residual| / |prediction|.
calibration_predictions = conformal["estimator"].predict(calibration_X)
normalised_scores = np.abs(calibration_y.to_numpy() - calibration_predictions) / np.maximum(
    np.abs(calibration_predictions), 1e-6
)
rank = int(np.ceil((len(normalised_scores) + 1) * 0.90))
normalised_quantile = float(np.sort(normalised_scores)[min(rank, len(normalised_scores)) - 1])
scale = np.maximum(np.abs(conformal["predictions"]), 1e-6)
lower_normalised = conformal["predictions"] - normalised_quantile * scale
upper_normalised = conformal["predictions"] + normalised_quantile * scale
normalised_report = prediction_interval_diagnostics(
    y_validation, lower_normalised, upper_normalised, alpha=0.10
)
print("normalised quantile (relative):", round(normalised_quantile, 4))
print("normalised coverage:", round(normalised_report["empirical_coverage"], 3),
      "| mean width:", round(normalised_report["mean_interval_width"], 2))

In [ ]:
def coverage_by_decile(lower, upper, labels="interval"):
    frame = pd.DataFrame(
        {
            "decile": pd.qcut(conformal["predictions"], 10, labels=False, duplicates="drop") + 1,
            "covered": (y_validation >= lower) & (y_validation <= upper),
            "width": upper - lower,
        }
    )
    table = frame.groupby("decile").agg(
        coverage=("covered", "mean"), mean_width=("width", "mean"), n=("covered", "size")
    )
    table.columns = pd.MultiIndex.from_product([[labels], table.columns])
    return table


coverage_table = pd.concat(
    [
        coverage_by_decile(conformal["lower"], conformal["upper"], "constant width"),
        coverage_by_decile(lower_normalised, upper_normalised, "normalised"),
    ],
    axis=1,
)
display(coverage_table.round(3))

figure, axes = plt.subplots(1, 2, figsize=(13, 4.4), constrained_layout=True)
order = np.argsort(conformal["predictions"])
axes[0].fill_between(
    np.arange(len(order)),
    conformal["lower"][order],
    conformal["upper"][order],
    color="C0", alpha=0.18, label="constant-width 90% interval",
)
axes[0].scatter(np.arange(len(order)), y_validation[order], s=8, color="C3", label="actual")
axes[0].set(xlabel="validation rows sorted by predicted value", ylabel="kWh/m2/yr",
            title="Conformal intervals (sorted)")
axes[0].legend(fontsize=8)

positions = np.arange(len(coverage_table))
axes[1].bar(positions - 0.2, coverage_table[("constant width", "coverage")], width=0.4,
            label=f"constant (overall {coverage_report['empirical_coverage']:.2f})")
axes[1].bar(positions + 0.2, coverage_table[("normalised", "coverage")], width=0.4,
            label=f"normalised (overall {normalised_report['empirical_coverage']:.2f})")
axes[1].axhline(0.90, color="k", linestyle="--", linewidth=1, label="nominal 0.90")
axes[1].set(xlabel="predicted-value decile", ylabel="empirical coverage",
            title="Marginal coverage is not conditional coverage")
axes[1].set_xticks(positions)
axes[1].set_xticklabels(coverage_table.index.astype(int))
axes[1].legend(fontsize=8, loc="lower right")
figure.suptitle("Two 90% intervals, two behaviours across the range", fontsize=13)
save_figure(figure, REPORT_DIR, "prediction-intervals")
display(figure)

## 13. Attribution

Permutation importance on the held-out partition, with R² as the scoring function. Two
cautions that matter more for regression than for classification:

* **a non-monotone effect has a small marginal importance by construction.** Averaging a
  curve that rises and falls cancels it. `insulation_score` is exactly that case here: the
  generator gives it a 9 kWh/m^2/yr sine interaction with glazing, yet the ranking puts it
  near the bottom, and the 2-D view below is the only honest way to read it;
* with a duplicated or correlated feature the drop is shared arbitrarily between the twins;
* the score is dominated by the high-demand rows, because R² weights squared error. That is
  the same weighting the model was trained under, which is consistent — but a stakeholder
  who cares about the efficient buildings is asking a different question.

In [ ]:
importance = tracker.permutation_importance(
    logged_id,
    reference_partition="validation",
    scoring="r2",
    n_repeats=5,
    max_samples=300,
    random_state=SEED,
)
importance_table = importance["importances"]
display(importance_table[["feature", "importance_mean", "importance_std"]].head(10))

plt.close("all")
figure = figure_of(
    tracker.plot_permutation_importance(
        logged_id,
        top_n=10,
        title="Permutation importance (validation, R2 drop)",
        reference_partition="validation",
        scoring="r2",
        n_repeats=5,
        max_samples=300,
        random_state=SEED,
    )
)
save_figure(figure, REPORT_DIR, "permutation-importance")
display(figure)

In [ ]:
# One-hot indicators are excluded from the curves: a partial-dependence line for a dummy
# is a two-point step, which is a table entry, not a curve.
indicator_columns = [column for column in feature_columns if column.startswith("heating_system_")]
top_features = [
    feature
    for feature in importance_table.nlargest(6, "importance_mean")["feature"]
    if feature not in indicator_columns
][:3]
plt.close("all")
figure = figure_of(
    tracker.plot_partial_dependence(
        logged_id,
        top_features,
        reference_partition="validation",
        kind="both",
        max_samples=300,
        ice_samples=60,
        grid_resolution=25,
        random_state=SEED,
        title="Partial dependence and ICE (top 3 features)",
    )
)
save_figure(figure, REPORT_DIR, "partial-dependence")
display(figure)

### A 2-D partial dependence for a non-monotone interaction

The one-dimensional curves above cannot show an effect that changes sign, so the pair that
the generator couples is plotted as a surface. The average prediction is recomputed on a
grid while every other feature keeps its observed value, in the same way
`PartialDependenceDisplay.from_estimator` does it — spelled out here because the point of
the figure is the *procedure*.

In [ ]:
grid = np.linspace(0.05, 1.0, 22)
reference_rows = validation[feature_columns].sample(
    min(250, len(validation)), random_state=SEED
)
surface = np.empty((grid.size, grid.size))
for row, glazing in enumerate(grid):
    for column, insulation in enumerate(grid):
        perturbed = reference_rows.copy()
        perturbed["insulation_score"] = insulation
        perturbed["glazing_score"] = glazing
        surface[row, column] = best_model.predict(perturbed).mean()

deviation = surface - surface.mean()
print(f"2-D PDP range: {deviation.min():.2f} .. {deviation.max():.2f} kWh/m2/yr from the mean")
print(f"1-D importance of insulation_score: {importance_table.set_index('feature').loc['insulation_score', 'importance_mean']:.4f} R2 drop")

figure, ax = plt.subplots(figsize=(5.8, 4.4), constrained_layout=True)
filled = ax.contourf(grid, grid, deviation, levels=15, cmap="RdBu_r")
ax.contour(grid, grid, deviation, levels=[0.0], colors="k", linewidths=0.8)
figure.colorbar(filled, ax=ax, label="kWh/m2/yr relative to the average")
ax.set(xlabel="insulation score", ylabel="glazing score",
       title="2-D partial dependence: insulation x glazing")
save_figure(figure, REPORT_DIR, "interaction-pdp")
display(figure)
print("The surface moves only a few kWh/m2 around the average, while the generator built")
print("in a far larger non-monotone term. That gap is itself the finding: a marginal ranking")
print("cannot see this effect, and a model fitted on 560 noisy rows only partly recovers it.")

## 14. Learning curve and monitoring

The learning curve answers "is more data the lever?". The monitoring cells answer "what
would tell us the model has gone stale?" — feature drift and, more directly for a
regression, **prediction distribution drift**: if the predicted demand level moves, the
model is being applied to a different building stock than it was fitted on.

In [ ]:
plt.close("all")
figure = figure_of(
    tracker.plot_learning_curve(
        logged_id,
        "Learning curve: AML-selected regressor",
        measure="r2",
        cv=KFold(n_splits=4, shuffle=True, random_state=SEED),
        train_sizes=np.linspace(0.3, 1.0, 5),
    )
)
save_figure(figure, REPORT_DIR, "learning-curve")
display(figure)

In [ ]:
serving = validation[feature_columns].copy()
serving["insulation_score"] = np.clip(serving["insulation_score"] * 0.75, 0, 1)
serving["degree_days_z"] = serving["degree_days_z"] + 1.2

drift = drift_report(train[feature_columns], serving, model_version="aml-regressor")
drift_rows = pd.DataFrame(
    [
        {
            "feature": feature,
            "psi": entry["value"],
            "status": "alert" if entry["value"] >= drift["thresholds"]["psi"] else "ok",
        }
        for feature, entry in drift["drift"]["numeric"].items()
    ]
)
display(drift_rows.nlargest(8, "psi"))

shifted_predictions = best_model.predict(serving)
prediction_drift = prediction_distribution_report(
    validation_predictions, shifted_predictions, model_version="aml-regressor"
)
entry = prediction_drift["drift"]["numeric"]["prediction"]
threshold = prediction_drift["thresholds"]["psi"]
print("prediction PSI:", round(entry["value"], 3),
      "|", "alert" if entry["value"] >= threshold else "ok",
      f"(threshold {threshold})")
print("prediction mean:", round(float(validation_predictions.mean()), 2),
      "->", round(float(shifted_predictions.mean()), 2))
print("prediction sd  :", round(float(validation_predictions.std(ddof=1)), 2),
      "->", round(float(shifted_predictions.std(ddof=1)), 2))
print("prediction-level alerts:", json.dumps(prediction_drift["alerts"], indent=1), "(PSI < 0.2)")
print("Feature drift and output drift disagree here: the shifted feature acts mostly")
print("through a non-linear term, so the distribution of predictions barely moves.")
print("Both views are watched, and neither substitutes for labels.")

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
top_drift = drift_rows.nlargest(8, "psi").sort_values("psi")
axes[0].barh(top_drift["feature"], top_drift["psi"],
             color=["C3" if value > 0.25 else "C1" if value > 0.1 else "C0" for value in top_drift["psi"]])
axes[0].axvline(0.1, color="k", linestyle=":", linewidth=0.9)
axes[0].axvline(0.25, color="k", linestyle="--", linewidth=0.9)
axes[0].set(xlabel="population stability index", title="Feature drift vs training reference")
axes[0].tick_params(axis="y", labelsize=8)

axes[1].hist(validation_predictions, bins=30, alpha=0.6, label="reference predictions", color="C0")
axes[1].hist(shifted_predictions, bins=30, alpha=0.6, label="shifted serving predictions", color="C3")
axes[1].set(xlabel="predicted kWh/m2/yr", ylabel="buildings", title="Prediction distribution shift")
axes[1].legend(fontsize=8)
figure.suptitle("Monitoring without labels", fontsize=13)
save_figure(figure, REPORT_DIR, "monitoring")
display(figure)

## 15. Package, gate, serve

The bundle records the metrics, the schema and the configuration. The conformal quantile is
stored in `metadata` rather than in the estimator, because it is a property of the
calibration protocol, not of the model — and because tabular serving emits point
predictions by contract. The decision layer adds the interval.

In [ ]:
schema = {column: {"dtype": str(train[column].dtype)} for column in feature_columns}
bundle = ModelBundle(
    best_model,
    task_name="notebook-regression",
    schema=schema,
    metrics={
        "r2": float(metric_table.loc[0, "r2"]),
        "mae": float(metric_table.loc[0, "mae"]),
        "rmse": float(metric_table.loc[0, "rmse"]),
        "mean_outer_score": float(evaluation["mean_outer_score"]),
    },
    owner="notebook",
    run_id="notebook-regression-run",
    dependencies={},
    metadata={
        "conformal": {
            "method": conformal["method"],
            "alpha": conformal["alpha"],
            "quantile": conformal["quantile"],
            "calibration_size": conformal["calibration_size"],
            "normalised_quantile": normalised_quantile,
            "empirical_coverage": coverage_report["empirical_coverage"],
            "normalised_empirical_coverage": normalised_report["empirical_coverage"],
        },
        "aml_audit": audit,
        "metric_intervals": interval_table.to_dict(orient="records"),
    },
)
bundle.record_audit_event(
    "training",
    estimator=audit["best_candidate"]["pipeline"],
    validation_rmse=float(metric_table.loc[0, "rmse"]),
)
bundle_path = save_bundle(bundle, workdir / "model.bundle")
loaded = load_bundle(bundle_path)

quality = assert_bundle_quality(
    bundle_path,
    expected_schema=schema,
    metric_thresholds={"r2": 0.60},
)
print("quality gate passed:", quality["passed"] if "passed" in quality else quality)

registry = ModelRegistry(workdir / "registry")
model_version = registry.register(loaded)
DeploymentApprovalGate(registry).promote(
    loaded.task_name, model_version, "production",
    approvers=["model-owner"], quality_report=quality,
)
print("promoted ->", model_version)

In [ ]:
request = validation[feature_columns].head(5).copy()
predictions_frame = predict_batch(
    loaded,
    request,
    output_path=workdir / "predictions.csv",
    request_id="notebook-regression-request",
)
point = predictions_frame["prediction"].to_numpy()
# The interval is applied by the decision layer from the stored calibration.
predictions_frame["lower"] = point - conformal["quantile"]
predictions_frame["upper"] = point + conformal["quantile"]
display(predictions_frame.round(2))
print("half-width (kWh/m2/yr):", round(conformal["quantile"], 3),
      "at 90% nominal coverage")

## 16. The report

In [ ]:
generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
overview = missing_overview(frame, target=TARGET, ignored=[])

missing_section = f"""
Complete rows: {overview['complete_rows']} of {overview['rows']} ({overview['complete_pct']:.1%}).
`setpoint_c` is MAR on the heating system: {setpoint_missing.set_index('heating_system')['mean'].round(3).to_dict()}.
Little's MCAR test on the pruned frame: statistic
{mechanism_pruned['mcar']['statistic']:.2f} on {mechanism_pruned['mcar']['df']} df,
p = {mechanism_pruned['mcar']['p_value']:.3f}.

{markdown_table(imputation_table, digits=3)}
"""

report_sections = [
    ("Summary", f"""
Target `{TARGET}` (kWh per m2 per year) on {len(frame)} buildings with {len(feature_columns)}
modelled features.

| estimate | value | interval | scope |
| --- | --- | --- | --- |
| nested-CV mean R2 | {evaluation['mean_outer_score']:.3f} | {interval['lower']:.3f}-{interval['upper']:.3f} | repeat variability |
| held-out R2 | {metric_table.loc[0, 'r2']:.3f} | {metric_intervals['r2'][0]:.3f}-{metric_intervals['r2'][1]:.3f} | row bootstrap |
| held-out MAE | {metric_table.loc[0, 'mae']:.2f} | {metric_intervals['mae'][0]:.2f}-{metric_intervals['mae'][1]:.2f} | kWh/m2/yr |
| held-out RMSE | {metric_table.loc[0, 'rmse']:.2f} | {metric_intervals['rmse'][0]:.2f}-{metric_intervals['rmse'][1]:.2f} | kWh/m2/yr |
| 90% conformal width | {coverage_report['mean_interval_width']:.2f} | - | constant width, marginal coverage {coverage_report['empirical_coverage']:.2f} |
"""),
    ("Data", f"""
Source: {dataset.source}. {dataset.description}. {len(frame)} rows, {len(feature_columns)}
modelled features after pruning one exact duplicate (`gross_area_m2`) and encoding the
heating system.

Condition number: {describe_condition(mcl['condition_number'])} before pruning,
{describe_condition(reprofiled['multicollinearity']['condition_number'])} after. Pruned
columns: `gross_area_m2` (exact duplicate of `floor_area_m2`) and {', '.join(f'`{c}`' for c in identifier_like) or 'none'} (identifier, unique on every row).
"""),
    ("Missing data", missing_section),
    ("Model selection", f"""
Metric: 4-fold CV RMSE on the training partition. Budget: {audit['budget']['max_evals']}
evaluations per family, {audit['budget']['time_limit']} s limit. Termination:
{audit['termination']['termination_reason']}. Selected: {', '.join(audit['best_candidate']['pipeline'])}.
Best CV RMSE {best_cv_rmse:.2f} kWh/m2/yr.

{markdown_table(trial_table.nsmallest(5, 'cv_rmse')[['family','trial','cv_rmse','duration_s']], digits=3)}
"""),
    ("Evaluation", f"""
{markdown_table(metric_table, digits=3)}

{markdown_table(interval_table, digits=3)}
"""),
    ("Residual diagnostics", f"""
The residual standard deviation rises with the predicted level (multiplicative noise):

{markdown_table(spread[['prediction_mean','residual_sd','residual_mae','n']], digits=3)}

Relative error by predicted decile:

{markdown_table(error_by_decile.reset_index(), digits=2)}
"""),
    ("Prediction intervals", f"""
Split conformal, alpha = {conformal['alpha']:.2f}, calibration on {conformal['calibration_size']}
training rows disjoint from the model fit. Constant half-width {conformal['quantile']:.2f};
empirical coverage {coverage_report['empirical_coverage']:.3f} on {coverage_report['sample_size']}
held-out rows, nominal {coverage_report['nominal_coverage']:.2f}.

Normalised variant: quantile {normalised_quantile:.4f} of the relative residual, coverage
{normalised_report['empirical_coverage']:.3f}, mean width {normalised_report['mean_interval_width']:.2f}.

{markdown_table(coverage_table.reset_index().assign(decile=lambda d: d['decile'].astype(int)), digits=3)}
"""),
    ("Explanation", f"""
Permutation importance, R2 drop, five repeats, held-out partition.

{markdown_table(importance_table[['feature','importance_mean','importance_std']].head(10), digits=4)}
"""),
    ("Monitoring", f"""
A serving frame was shifted (insulation x0.75, degree days +1.2 sd) to show what the
monitoring tells you before labels arrive.

{markdown_table(drift_rows.nlargest(6, 'psi'), digits=3)}
"""),
    ("Limitations", f"""
Synthetic data; small search budget; one 70/30 split. Bootstrap intervals describe sampling
variability of this partition, not next year's stock. Conformal coverage is marginal under
exchangeability: the decile table shows it is not conditional, which matters for decisions
taken on high-demand buildings. Permutation importance is attribution on this model and this
reference set, and is shared arbitrarily between correlated features. Nothing here is a
causal statement about insulation, age or heating systems.
"""),
]

report_markdown = "# Regression report: building energy intensity\n\n" + "\n\n".join(
    f"## {title}\n\n{body.strip()}" for title, body in report_sections
)
report_markdown += "\n\nGenerated at " + generated_at + " from a seeded, reproducible notebook run.\n"

tables = {
    "profile-columns": columns_table,
    "target-associations": association_table,
    "missing-shifts": pd.DataFrame(mechanism["shifts"]),
    "imputation-benchmark": imputation_table,
    "leakage-comparison": comparison,
    "search-trials": trial_table.drop(columns=["params"]),
    "metrics": metric_table,
    "metric-intervals": interval_table,
    "residual-spread": spread,
    "error-by-decile": error_by_decile.reset_index(),
    "conformal-coverage": coverage_table.reset_index(),
    "permutation-importance": importance_table[["feature", "importance_mean", "importance_std"]],
    "drift": drift_rows,
}
for name, table in tables.items():
    table.to_csv(REPORT_DIR / f"{name}.csv", index=False)

(REPORT_DIR / "analysis-metadata.json").write_text(
    json.dumps(
        {
            "generated_at": generated_at,
            "dataset": {"source": dataset.source, "description": dataset.description, "notes": dataset.notes},
            "seed": SEED,
            "target": TARGET,
            "features": feature_columns,
            "rows": {"train": len(train), "validation": len(validation)},
            "metrics": metric_table.to_dict(orient="records"),
            "metric_intervals": interval_table.to_dict(orient="records"),
            "conformal": bundle.metadata["conformal"],
            "search": {"budget": audit["budget"], "termination": audit["termination"],
                        "best": audit["best_candidate"]},
            "evaluation": {
                "mean_outer_score": evaluation["mean_outer_score"],
                "confidence_interval": evaluation["confidence_interval"],
                "repeated_outer_scores": evaluation["repeated_outer_scores"],
            },
            "model_version": model_version,
            "limitations": [
                "Synthetic data; small search budget; single split.",
                "Conformal coverage is marginal, not conditional.",
                "Permutation importance is attribution, not causation.",
            ],
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

report_figures = {}
for stem in (
    "eda-overview",
    "missing-mechanism",
    "imputation-benchmark",
    "search-trace",
    "evaluation",
    "residual-diagnostics",
    "prediction-intervals",
    "permutation-importance",
    "interaction-pdp",
    "monitoring",
):
    pdf_path = REPORT_DIR / f"{stem}.pdf"
    if pdf_path.exists():
        report_figures[f"figures/{stem}.pdf"] = pdf_path.read_bytes()

from api.reports import figure_latex

inserts = {
    ("Data", None): figure_latex("figures/eda-overview.pdf", "Target shape, missingness and marginal associations."),
    ("Missing data", None): figure_latex("figures/missing-mechanism.pdf", "Missingness of the setpoint by heating system."),
    ("Model selection", None): figure_latex("figures/search-trace.pdf", "Search trace and best-so-far against cumulative fit time."),
    ("Evaluation", None): figure_latex("figures/evaluation.pdf", "Nested-CV repeats, interval and held-out estimate."),
    ("Residual diagnostics", None): figure_latex("figures/residual-diagnostics.pdf", "Predicted vs actual, heteroscedastic residuals, Q-Q, error by decile."),
    ("Prediction intervals", None): figure_latex("figures/prediction-intervals.pdf", "Conformal intervals and coverage by predicted decile."),
    # figure_latex returns a list of LaTeX lines, so several figures under one heading
    # are concatenated -- two dict entries with the same key would silently keep one.
    ("Explanation", None): (
        figure_latex("figures/permutation-importance.pdf", "Permutation importance on the held-out partition.")
        + figure_latex("figures/interaction-pdp.pdf", "2-D partial dependence of the insulation-glazing pair.")
    ),
    ("Monitoring", None): figure_latex("figures/monitoring.pdf", "Feature drift and prediction distribution shift."),
}

report_path = REPORT_DIR / "report.md"
report_path.write_text(report_markdown, encoding="utf-8")
pdf, note = compile_pdf(
    report_markdown,
    task="notebook-regression",
    title="Building energy intensity: end-to-end regression",
    subtitle=f"SKSurrogate {SKSurrogate.__version__} / seed {SEED}",
    figures=report_figures,
    inserts=inserts,
    generated_at=generated_at,
)
if pdf is not None:
    (REPORT_DIR / "report.pdf").write_bytes(pdf)
print("PDF:", note)
print("artifacts:", sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))

In [ ]:
from IPython.display import Markdown

Markdown(report_markdown)

## What this notebook does *not* establish

* **No causal reading of the drivers.** Insulation, age and heating system are correlated in
  the generator by construction; the partial-dependence curves describe this model, not a
  retrofit policy.
* **The interval is marginal.** Conformal coverage holds on average under exchangeability.
  The decile table shows exactly where that promise is weakest.
* **One split, one seed.** The bootstrap intervals cover sampling variability of this
  partition; a different 70/30 split would move the numbers within roughly that range, not
  outside it.
* **R² weights the expensive buildings.** A model tuned for relative error would look
  different and might serve a different decision better.
* **The residuals are not Gaussian**, and the Q-Q panel exists to show that rather than to
  justify a normal-theory interval.

Next: [03_time_series_end_to_end.ipynb](03_time_series_end_to_end.ipynb) replaces the
random split with a rolling-origin backtest, the accuracy metric with MASE against naive
baselines, and the conformal regression interval with a horizon-aware forecast interval.

In [ ]:
tracker.close()
print("done.")